# Nozzle inverse by QSVT

Fit fresh phase sequences and solve both original nozzle systems with FlagQuantum.

## Packed simulator operations are explicit implementation choices

The displayed Circuit builder packs the two projector phase streams into
an exact simulator matrix and composes it with input preparation and H gates.
FlagQuantum executes that Circuit; host code fits degree-101 and degree-255
inverse sequences and sets the Wx-to-block phase convention. This packed
operation is not a gate-level hardware synthesis claim or a single native
qfunc linear-solver API.

Inspect complete-state error, reciprocal response and selector-zero solution
rescaling separately. Try another source right-hand side with the same field
layout and verify its independent stream evaluation. Build and execution
times include the declared dense host work. Conditional projection probability
and classical solution error must remain visible; normalized shape alone is
insufficient to validate a quantum inverse solver.


Use the local FlagQuantum environment.

In [ ]:
import flagquantum as fq
import torch

from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.workflow import (
    CTYPE,
    Encoding,
    branch_matrix,
    preparation,
    projector_signs,
)


def circuit_preview(
    enc: Encoding, block: torch.Tensor, phases: torch.Tensor, rhs: torch.Tensor
) -> fq.Circuit:
    """Apply the two projector sequences as exact packed simulator operations."""
    encoded_qubits = enc.data_width + enc.block_width
    data = tuple(range(encoded_qubits + 1 - enc.data_width, encoded_qubits + 1))
    signs = projector_signs(block, rhs.numel())
    direct = branch_matrix(block, signs, phases)
    opposite = branch_matrix(block, signs, -phases)
    size = block.size(0)
    branches = torch.zeros((2 * size, 2 * size), dtype=CTYPE)
    branches[:size, :size] = direct
    branches[size:, size:] = opposite
    circuit = fq.Circuit(encoded_qubits + 1, dtype=CTYPE)
    circuit.compose(preparation(rhs / rhs.norm()), qubits=data)
    circuit.h(0)
    circuit.any(*range(encoded_qubits + 1), matrix=branches)
    return circuit.h(0)


from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq


Import the native QSVT solve.

In [ ]:
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.model import (
    source_matrix,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.main import (
    print_report,
    save_figure,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_chebyshev_lcu.overview.workflow import (
    input_vector,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.workflow import (
    solve,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.model import (
    source_matrix as small_matrix,
)

print(fq.__name__)


Fit each odd inverse polynomial and apply its alternating block and adjoint sequence.

In [ ]:
print("Original inverse polynomial degrees:", 101, 255)


Select the CPU simulator for the original-size examples.

In [ ]:
print("Execution target: local CPU")


Choose the output figure directory.

In [ ]:
figures = Path("notebook-figures")


Read the original eight-row system and right-hand side.

In [ ]:
matrix_small = small_matrix()
rhs_small = input_vector("b_nozzle_small", 3)
print(matrix_small.shape, rhs_small.shape)


Regenerate 102 phases and execute the degree-101 Pauli solve.

In [ ]:
small = solve(False)
print_report("Small", small)


Compare the postselected small solution with direct linear algebra.

In [ ]:
save_figure(small, "Small nozzle inverse", figures / "nozzle-small.png")
display(Image(filename=str(figures / "nozzle-small.png")))


Check the original small-system error limit.

In [ ]:
assert small.solution_l2 < 0.1


Read the original sixteen-row system and right-hand side.

In [ ]:
matrix_large = source_matrix()
rhs_large = input_vector("nozzle_008_b", 4)
print(matrix_large.shape, rhs_large.shape)


Keep all banded block and data bits.

In [ ]:
print("Banded data width:", matrix_large.size(0).bit_length() - 1)


Regenerate 256 phases and execute the degree-255 banded solve.

In [ ]:
large = solve(True)
print_report("Large", large)


Compare the postselected large solution with the dense reference.

In [ ]:
save_figure(large, "Large nozzle inverse", figures / "nozzle-large.png")
display(Image(filename=str(figures / "nozzle-large.png")))


Check the original large-system error limit.

In [ ]:
assert large.solution_l2 < 0.1
